# Genizah focus v1.9c — FULL-WEIGHT merger (v19a recipe, merging layers train unconstrained)

ONE variable vs **v19b**: the 8 merging linears (`visual.merger.linear_fc{1,2}` +
`visual.deepstack_merger_list.{0,1,2}.linear_fc{1,2}`, ~160M params) train
**full-weight via PEFT `modules_to_save`** instead of r16 LoRA. Everything else
is byte-identical to v19b: same data pins, mixture, v18b step-700 warm start,
2000-step cosine at 5e-5 (schedule shape kept so per-step comparisons hold).

Why: v19b's merger LoRA updates SATURATE the r16 budget (rank-for-90%-energy
10-11/16, flattest spectra in the adapter) and the 2026-08-30 ablation probe
showed the merger update is behaviorally potent — it alone caused the worst
multi-col collapse AND real abstention→read rescues, and v19b's tower/language
co-adapted with it (ablated model abstains on 11% of PGP). Hypothesis: full
rank keeps the rescues and drops the collapses.

**Run plan:** kill at ~step 700 (half the units); compare step-700 religious
(GT rev 1.1) + PGP-131 vs v19b-700/v19a. **Tripwire:** eval/loss@100 must be
≈0.78 (v19b: 0.7824). If > 0.82 the full-weight merger is destabilizing at
5e-5 — stop and restart with learning_rate=2e-5.

Gates differ from v19b in kind: merger modules must be UN-QUANTIZED to clone,
they start EQUAL to base (identity), grads flow to the 8 full weights, and the
export requires them to have MOVED AWAY from base.


In [ ]:
# Cell 1 — installs + env (PINNED: the exact triplet audited 2026-08-09;
# unpinned installs float and transformers 5.x would invalidate the vision-path
# analysis AND the 108-tensor count)
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["WANDB_PROJECT"] = "qwen-hebrew-finetune"
%pip install -q "unsloth[colab-new]==2026.8.9" "unsloth_zoo==2026.8.6" "transformers==4.57.6" hf_transfer wandb

from importlib.metadata import version
assert version("unsloth_zoo") == "2026.8.6", f"unsloth_zoo drifted: {version('unsloth_zoo')}"
assert version("transformers") == "4.57.6", f"transformers drifted: {version('transformers')}"

import torch
assert torch.cuda.is_available(), "No GPU — switch runtime to A100"

In [ ]:
# Cell 2 — data: KTIV corpus + genizah_clean_v2 + synthetic v3 + talmud replay
# (all pinned — IDENTICAL to v19a: the data is not a variable in this run)
from google.colab import userdata
from huggingface_hub import login, snapshot_download
from datasets import load_dataset

login(token=userdata.get("HF_TOKEN"))

GENIZAH_REPO = "isaacmg/genizah_clean_v2"
GENIZAH_REVISION = "57366ad378946918731ad0012d699acc7d9ed31c"  # 1,055/56 repaired+screened
KTIV_REPO = "isaacmg/genizah_ktiv_v1"
KTIV_REVISION = "ccf3a25ecd39da3e444d453310ac7169dfd603f8"     # 914 mss / 8,298 train rows
SYNTH3_REPO = "isaacmg/synthetic_hebrew_v3"
SYNTH3_REVISION = "59abcf7c30fb6753b9df89f0a099b07a68059013"  # 12 faces, probe-weighted drills
TALMUD_REPO = "isaacmg/talmud_finetune_v2"

for _sha in (GENIZAH_REVISION, KTIV_REVISION, SYNTH3_REVISION):
    assert len(_sha) == 40, f"unpinned revision: {_sha!r}"

genizah = load_dataset(GENIZAH_REPO, split="train", revision=GENIZAH_REVISION)
genizah_val = load_dataset(GENIZAH_REPO, split="val", revision=GENIZAH_REVISION)
ktiv = load_dataset(KTIV_REPO, split="train", revision=KTIV_REVISION)
ktiv_val = load_dataset(KTIV_REPO, split="val", revision=KTIV_REVISION)
synth3 = load_dataset(SYNTH3_REPO, split="train", revision=SYNTH3_REVISION)
synth3_eval = load_dataset(SYNTH3_REPO, split="eval", revision=SYNTH3_REVISION)
talmud = load_dataset(TALMUD_REPO, split="train")
talmud_val = load_dataset(TALMUD_REPO, split="val")

# KTIV task families (page rows are the bulk per the v1.9 plan)
ktiv_pages = ktiv.filter(lambda t: t == "fragment_transcribe", input_columns="task")
ktiv_regions = ktiv.filter(lambda t: t == "region_transcribe", input_columns="task")
ktiv_crops = ktiv.filter(
    lambda t: t in ("section_transcribe", "line_transcribe"), input_columns="task")

# Talmud replay (forgetting protection; same v16-lesson exclusions as v18)
crops = talmud.filter(lambda t: t == "crop_transcribe", input_columns="task")
pages = talmud.filter(lambda t: t == "page_extract", input_columns="task")
gemara_crops = crops.filter(lambda s: s == "gemara", input_columns="section")
pages_small = pages.filter(
    lambda s: s in ("rashi", "tosafot"), input_columns="section")
_n_before = len(pages_small)
pages_small = pages_small.filter(
    lambda st: st.split("_")[0] not in ("16", "05"), input_columns="stem")
print(f"pages_small: {_n_before} -> {len(pages_small)} after vol 16/05 exclusion")
assert len(pages_small) < _n_before, "vol 16/05 exclusion filtered nothing"

print(f"ktiv: pages={len(ktiv_pages)} regions={len(ktiv_regions)} crops={len(ktiv_crops)} | "
      f"genizah={len(genizah)} synth3={len(synth3)} "
      f"pages_small={len(pages_small)} gemara_crops={len(gemara_crops)}")
assert len(ktiv_pages) > 1200 and len(ktiv_regions) > 1500 and len(ktiv_val) >= 300
assert len(genizah) > 1000 and len(genizah_val) >= 50
assert len(synth3) > 10000 and len(synth3_eval) >= 800

# Label hygiene on BOTH genizah sources: no internal gap tokens or mojibake
for _name, _ds in (("genizah", genizah), ("ktiv", ktiv_pages)):
    _sample = _ds.shuffle(seed=0).select(range(200))["answer"]
    assert all("␣" not in a for a in _sample), f"{_name}: internal gap token leaked"
    assert all(not any(ch in a for ch in "&#$_{}<>\\") for a in _sample), f"{_name}: mojibake leaked"
    assert any("[...]" in a for a in _sample), f"{_name}: expected damage-gap markers"
# KTIV region rows must carry the restriction instruction
_q = ktiv_regions.shuffle(seed=0).select(range(50))["question"]
assert all("Transcribe ONLY" in q for q in _q), "region rows lost their restriction"

In [ ]:
# Cell 3 — model at page resolution + FULL-WEIGHT merger + v18b step-700 warm start
from unsloth import FastVisionModel
from unsloth_zoo.peft_utils import get_peft_regex
from transformers import AutoImageProcessor

MAX_SEQ = 12288
# Global 6.5MP floor (train/infer resolution contract, ships in the export).
MIN_PIX = 6_500_000
MAX_PIX = 7_000_000

WARM_CKPT_REPO = "isaacmg/qwen3-vl-8b-hebrew-v18b-ckpt"
WARM_REVISION = "c80313f8208558df5fd774be257b146c4b4749d6"  # step 700, eval_loss 0.473

TRAIN_VISION_LORA = True    # unchanged from v19a
MERGER_FULL_WEIGHT = True   # THE v1.9c variable: merging layers train full-weight
MERGER_MODULES = [
    "visual.merger.linear_fc1", "visual.merger.linear_fc2",
    *[f"visual.deepstack_merger_list.{i}.linear_fc{j}"
      for i in range(3) for j in (1, 2)],
]

model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen3-VL-8B-Instruct",
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
    max_seq_length=MAX_SEQ,
)

# v19a's target set, computed by the same helper unsloth uses internally when
# no explicit target_modules is given. The merger is NOT in the LoRA targets —
# it trains full-weight through modules_to_save (PEFT clones each module and
# trains the clone; the original stays as the frozen reference).
base_regex = get_peft_regex(
    model,
    finetune_vision_layers=True, finetune_language_layers=True,
    finetune_attention_modules=True, finetune_mlp_modules=True,
)

# modules_to_save cannot train a quantized clone: the 8 merger linears must
# have arrived UN-quantized (unsloth's 4-bit skip-list normally leaves the
# vision stack in bf16). Hard gate with remediation, not a silent fallback.
import bitsandbytes as bnb
_mods = {n: m for n, m in model.named_modules()
         if any(n.endswith(s) for s in MERGER_MODULES)}
assert len(_mods) == 8, f"merger modules found: {sorted(_mods)}"
_quantized = [n for n, m in _mods.items()
              if isinstance(m, (bnb.nn.Linear4bit, bnb.nn.Linear8bitLt))]
assert not _quantized, (
    f"merger modules are quantized: {_quantized} — modules_to_save would clone "
    "4-bit weights (untrainable). Reload with these modules in "
    "llm_int8_skip_modules, or load_in_4bit=False, before proceeding.")

model = FastVisionModel.get_peft_model(
    model,
    target_modules=base_regex,
    modules_to_save=MERGER_MODULES if MERGER_FULL_WEIGHT else None,
    finetune_vision_layers=True, finetune_language_layers=True,
    finetune_attention_modules=True, finetune_mlp_modules=True,
    r=16, lora_alpha=16, lora_dropout=0.0, bias="none", random_state=3407,
)

# the experiment is void unless tower LoRA exists AND the merger became
# 8 trainable full-weight clones (and got NO LoRA — that would be v19b)
_tower_lora = [n for n, _ in model.named_parameters()
               if ".visual.blocks." in n and "lora_A" in n]
_merger_lora = [n for n, _ in model.named_parameters()
                if "merger" in n and "lora_A" in n]
_merger_full = [n for n, p in model.named_parameters()
                if "merger" in n and ".modules_to_save." in n
                and n.endswith(".weight") and p.requires_grad]
assert len(_tower_lora) == 108, f"tower adapters: {len(_tower_lora)}/108"
assert not _merger_lora, f"merger got LoRA (this is v19b, not v19c): {_merger_lora[:2]}"
if MERGER_FULL_WEIGHT:
    assert len(_merger_full) == 8, (
        f"merger full-weight clones: {len(_merger_full)}/8 — unsloth did not "
        "pass modules_to_save through to PEFT; do NOT train (this would "
        "silently replicate v19a)")
print(f"adapters: tower {len(_tower_lora)}/108 LoRA, "
      f"merger {len(_merger_full)}/8 full-weight")

# weights-only warm start (fresh optimizer + schedule) — same state as v19a
assert len(WARM_REVISION) == 40, "invalid revision SHA"
from safetensors.torch import load_file
from peft import set_peft_model_state_dict
local = snapshot_download(WARM_CKPT_REPO, revision=WARM_REVISION,
                          allow_patterns="last-checkpoint/adapter_model.safetensors")
missing = set_peft_model_state_dict(
    model, load_file(f"{local}/last-checkpoint/adapter_model.safetensors"))
print("unexpected keys:", len(getattr(missing, "unexpected_keys", [])))
# the warm start must arrive with a LIVE tower adapter (v18b property)
_wv = [n for n, p in model.named_parameters()
       if ".visual.blocks." in n and "lora_B" in n and p.detach().abs().max().item() > 0]
assert len(_wv) == 108, f"warm start vision adapter not loaded: {len(_wv)}/108 nonzero"
# v18b has no merger keys, so each merger clone must START EQUAL to its
# frozen original — v19c begins from the exact function v19a started from.
_wrappers = [(n, m) for n, m in model.named_modules()
             if any(n.endswith(s) for s in MERGER_MODULES)
             and hasattr(m, "modules_to_save")]
assert len(_wrappers) == 8, f"modules_to_save wrappers: {len(_wrappers)}/8"
for _n, _w in _wrappers:
    _clone = _w.modules_to_save["default"].weight
    _orig = _w.original_module.weight
    assert torch.allclose(_clone.detach().float(), _orig.detach().float()), (
        f"merger clone differs from base at start: {_n}")
print("warm start OK: tower 108/108 nonzero, merger 8/8 clones at identity")

if TRAIN_VISION_LORA:
    # Unsloth's requires_grad_for_gradient_checkpointing never matches the
    # Qwen3-VL vision tower (`enumerate(self.blocks)` loop; forward calls
    # get_image_features, not self.visual), so hidden states entering the
    # checkpointed vision blocks never require grad and UnslothCheckpointFunction
    # skips their backward entirely (verified against unsloth_zoo 2026.8.6).
    # Making the patch-embed output require grad restores gradient flow.
    def _vision_embeds_require_grad(module, inputs, output):
        if not torch.is_grad_enabled():
            return output
        output.requires_grad_(True)
        return output
    _patch_embed = next(m for n, m in model.named_modules()
                        if n.endswith("visual.patch_embed"))
    _patch_embed.register_forward_hook(_vision_embeds_require_grad)
    print("vision LoRA gradient fix: ON")

tokenizer.image_processor = AutoImageProcessor.from_pretrained(
    "unsloth/Qwen3-VL-8B-Instruct", min_pixels=MIN_PIX, max_pixels=MAX_PIX,
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
lora_params = [n for n, p in model.named_parameters()
               if p.requires_grad and "lora" in n.lower()]
assert trainable > 0 and lora_params
print(f"trainable: {trainable/1e6:.1f}M ({len(lora_params)} LoRA tensors) — "
      "NOTE: requires_grad-based count; the grad-flow cell below is the real check")
print("resolution:", tokenizer.image_processor.size)

In [ ]:
# Cell 4 — conversation format + collator (native res preserved by resize='max')
def to_conversation(sample):
    return {
        "messages": [
            {"role": "user", "content": [
                {"type": "image", "image": sample["image"]},
                {"type": "text", "text": sample["question"]},
            ]},
            {"role": "assistant", "content": [
                {"type": "text", "text": sample["answer"]},
            ]},
        ]
    }

from unsloth.trainer import UnslothVisionDataCollator

collator = UnslothVisionDataCollator(
    model, tokenizer,
    formatting_func=to_conversation,
    resize="max",
    max_seq_length=MAX_SEQ,
    train_on_responses_only=True,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

# guardrail: one real batch must show page-res pixels + masked labels
batch = collator([ktiv_pages[0], synth3[0]])
pv = batch["pixel_values"]
assert pv is not None and pv.shape[0] > 40000, (
    f"{pv.shape[0]} patch rows — expected >40k for two ~6.5MP images; "
    "the resolution policy is not reaching the collator")
labels = batch["labels"]
unmasked = labels[0][labels[0] != -100]
_tok = tokenizer.tokenizer if hasattr(tokenizer, "tokenizer") else tokenizer
assert ktiv_pages[0]["answer"][:30] in _tok.decode(unmasked)
print(f"collator OK: pixel rows={pv.shape[0]}, labels masked")

In [ ]:
# Cell 5 — gradient-flow verification (cheap; run BEFORE burning GPU-hours)
# Language LoRA must always receive gradients; tower LoRA iff TRAIN_VISION_LORA;
# merger FULL WEIGHTS iff MERGER_FULL_WEIGHT. HARD GATE, per-tensor (min) nonzero.
# The merger sits AFTER the checkpointed tower blocks (loss -> LLM -> merger
# -> blocks), so its grads do not depend on the patch_embed hook — if the
# 8-tensor check fails here, a DeepStack path is being skipped by unsloth's
# checkpointing and needs its own require-grad hook: STOP and investigate;
# training anyway would silently replicate v19a and void the control.
FastVisionModel.for_training(model)
_vb = {k: (v.to(model.device) if torch.is_tensor(v) else v)
       for k, v in collator([genizah[0]]).items()}
model(**_vb).loss.backward()
tower_g = [p.grad.abs().max().item() for n, p in model.named_parameters()
           if ".visual.blocks." in n and "lora_B" in n and p.grad is not None]
mrg_g = [p.grad.abs().max().item() for n, p in model.named_parameters()
         if "merger" in n and ".modules_to_save." in n and n.endswith(".weight")
         and p.grad is not None]
lang_g = [p.grad.abs().max().item() for n, p in model.named_parameters()
          if ".visual." not in n and "lora_B" in n and p.grad is not None]
model.zero_grad(set_to_none=True)
assert lang_g and max(lang_g) > 0, "language LoRA got no gradients"
if TRAIN_VISION_LORA:
    assert len(tower_g) == 108 and min(tower_g) > 0, (
        f"tower LoRA dead or partial: {len(tower_g)}/108 tensors with grads, "
        f"min|g|={min(tower_g) if tower_g else 0:.2e}")
if MERGER_FULL_WEIGHT:
    assert len(mrg_g) == 8 and min(mrg_g) > 0, (
        f"merger full weights dead or partial: {len(mrg_g)}/8 tensors with "
        f"grads, min|g|={min(mrg_g) if mrg_g else 0:.2e}")
    print(f"merger FULL-WEIGHT training CONFIRMED: 8/8 grads, "
          f"min|g|={min(mrg_g):.2e} max|g|={max(mrg_g):.2e}")
print(f"tower min|g|={min(tower_g):.2e}  language max|g|={max(lang_g):.2e}")

In [ ]:
# Cell 6a — OPTIONAL throughput benchmark (flag-gated; run once, then set False)
# The A100 ran v18 at batch 1x8 with 13.8/40GB VRAM — headroom. This times
# fwd+bwd for candidate batch geometries at the REAL resolution so the long
# run uses the fastest safe config. To compare gradient-checkpointing modes
# ("unsloth" vs True), change it in cell 3 and rerun cells 3-6a once each.
RUN_THROUGHPUT_BENCH = False
BATCH_GEOMETRIES = [(1, 8), (2, 4), (4, 2)]   # (per_device_batch, grad_accum)

if RUN_THROUGHPUT_BENCH:
    import time
    FastVisionModel.for_training(model)
    bench_rows = [ktiv_pages[i] for i in range(8)] + [synth3[i] for i in range(8)]
    for bs, ga in BATCH_GEOMETRIES:
        try:
            torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
            t0 = time.time(); n_micro = 0
            for step in range(2):                    # 2 optimizer steps
                for micro in range(ga):
                    rows = [bench_rows[(n_micro + i) % len(bench_rows)] for i in range(bs)]
                    b = {k: (v.to(model.device) if torch.is_tensor(v) else v)
                         for k, v in collator(rows).items()}
                    (model(**b).loss / ga).backward()
                    n_micro += 1
                model.zero_grad(set_to_none=True)
            dt = (time.time() - t0) / 2
            peak = torch.cuda.max_memory_allocated() / 1e9
            print(f"batch {bs}x{ga}: {dt:.1f}s/optimizer-step, peak {peak:.1f}GB")
        except torch.cuda.OutOfMemoryError:
            print(f"batch {bs}x{ga}: OOM — skip")
            torch.cuda.empty_cache()
    model.zero_grad(set_to_none=True)
    print("pick the fastest non-OOM geometry and set it in cell 6, then set "
          "RUN_THROUGHPUT_BENCH = False")

In [ ]:
# Cell 6 — mixture + per-domain eval + training (resumable, max_steps-bounded)
# IDENTICAL to v19b except the checkpoint repo / run name: same mixture, same
# eval slices, same 2000-step cosine at 5e-5 (schedule SHAPE preserved so the
# first 700 steps compare per-step against v19b; plan is to KILL at ~700).
# TRIPWIRE: eval/loss@100 ≈ 0.78 expected (v19b 0.7824); if > 0.82 the merger
# full weights are destabilizing at 5e-5 — stop, restart at learning_rate=2e-5.
import inspect
import wandb
from datasets import concatenate_datasets, interleave_datasets
from huggingface_hub import list_repo_files
from trl import SFTConfig, SFTTrainer
from unsloth import is_bf16_supported

CKPT_REPO = "isaacmg/qwen3-vl-8b-hebrew-v19c-ckpt"  # private, auto-created
OUT_DIR = "outputs_v19c"
wandb.login(key=userdata.get("WANDB_API_KEY"))

# v1.9 mixture (unchanged): 30% ktiv pages / 20% genizah_clean_v2 / 20% synth3 /
# 8% ktiv regions / 7% ktiv crops / 10% talmud small-script pages / 5% gemara crops
mixture = interleave_datasets(
    [ktiv_pages, genizah, synth3, ktiv_regions, ktiv_crops,
     pages_small, gemara_crops],
    probabilities=[0.30, 0.20, 0.20, 0.08, 0.07, 0.10, 0.05], seed=3407,
    stopping_strategy="all_exhausted",
)
# val loss every 100 steps across every domain in the mixture
eval_ds = concatenate_datasets([
    ktiv_val.filter(lambda t: t == "fragment_transcribe", input_columns="task")
            .select(range(40)),
    genizah_val.select(range(40)),
    ktiv_val.filter(lambda t: t == "region_transcribe", input_columns="task")
            .select(range(15)),
    talmud_val.filter(lambda t: t == "page_extract", input_columns="task")
              .select(range(30)),
    talmud_val.filter(
        lambda t, s: t == "crop_transcribe" and s == "gemara",
        input_columns=["task", "section"]).select(range(15)),
    synth3_eval.select(range(20)),
])

def make_sft_config(**kw):
    params = inspect.signature(SFTConfig.__init__).parameters
    if "max_seq_length" in kw and "max_seq_length" not in params:
        kw["max_length"] = kw.pop("max_seq_length")
    dropped = {k: kw.pop(k) for k in list(kw) if k not in params}
    if dropped:
        print(f"⚠️ dropped unsupported SFTConfig kwargs: {sorted(dropped)}")
    return SFTConfig(**kw)

def make_trainer(**kw):
    try:
        return SFTTrainer(**kw)
    except TypeError as e:
        if "tokenizer" in kw and ("tokenizer" in str(e) or "processing_class" in str(e)):
            kw["processing_class"] = kw.pop("tokenizer")
            return SFTTrainer(**kw)
        raise

resume_dir = None
try:
    # hub_strategy="checkpoint" pushes a rolling "last-checkpoint/" folder
    files = list_repo_files(CKPT_REPO)
    if any(f.startswith("last-checkpoint/") for f in files):
        snapshot_download(CKPT_REPO, allow_patterns="last-checkpoint/*",
                          local_dir=OUT_DIR)
        resume_dir = f"{OUT_DIR}/last-checkpoint"
        print("resuming from last-checkpoint")
except Exception as e:
    print(f"no checkpoint repo yet ({type(e).__name__}) — fresh start")

FastVisionModel.for_training(model)
trainer = make_trainer(
    model=model, tokenizer=tokenizer, data_collator=collator,
    train_dataset=mixture, eval_dataset=eval_ds,
    args=make_sft_config(
        per_device_train_batch_size=1, gradient_accumulation_steps=8,
        max_steps=2000,                    # kill-anytime, resumes
        learning_rate=5e-5,                # continuation LR (same as v19a)
        warmup_ratio=0.02, lr_scheduler_type="cosine", weight_decay=0.01,
        logging_steps=10,
        eval_strategy="steps", eval_steps=100, per_device_eval_batch_size=1,
        save_steps=100, save_total_limit=2,
        push_to_hub=True, hub_model_id=CKPT_REPO,
        hub_strategy="checkpoint", hub_private_repo=True,
        optim="adamw_8bit", seed=3407, output_dir=OUT_DIR,
        report_to="wandb", run_name="genizah_focus_v19c",
        bf16=is_bf16_supported(), fp16=not is_bf16_supported(),
        remove_unused_columns=False, dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
        max_seq_length=MAX_SEQ,
    ),
)
trainer.train(resume_from_checkpoint=resume_dir)

In [ ]:
# Cell 7 — export merged (policy-carrying) model, gated on tower LoRA + merger MOVEMENT
MERGED_REPO = "isaacmg/qwen3-vl-8b-hebrew-v19c-merged"
if TRAIN_VISION_LORA:
    # shipped-adapter gate: the run is not a vision run unless the weights moved
    _nz = [n for n, p in model.named_parameters()
           if ".visual.blocks." in n and "lora_B" in n and p.detach().abs().max().item() > 0]
    assert len(_nz) == 108, f"shipped adapter tower lora_B nonzero: {len(_nz)}/108"
if MERGER_FULL_WEIGHT:
    _moved = []
    for _n, _m in model.named_modules():
        if any(_n.endswith(s) for s in MERGER_MODULES) and hasattr(_m, "modules_to_save"):
            _c = _m.modules_to_save["default"].weight.detach().float()
            _o = _m.original_module.weight.detach().float()
            if not torch.allclose(_c, _o):
                _moved.append(_n)
    assert len(_moved) == 8, (
        f"merger full weights moved: {len(_moved)}/8 — the merger never left "
        "identity; this is NOT a v1.9c run, do not export it as one")
    print("shipped checks: tower 108/108 LoRA moved, merger 8/8 full weights moved")
model.save_pretrained_merged("v19c-merged", tokenizer, save_method="merged_16bit")
# ship the TRAINING resolution policy with the model (hard rule)
tokenizer.image_processor.save_pretrained("v19c-merged")
model.push_to_hub_merged(MERGED_REPO, tokenizer, save_method="merged_16bit", private=True)
print("pushed", MERGED_REPO)